# Graph autoencoders — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def sigmoid(x): return 1/(1+np.exp(-x))
def softmax(x):
    x=np.asarray(x,dtype=float); e=np.exp(x-x.max()); return e/e.sum()
def draw_graph(A, pos=None, values=None, title=''):
    A=np.asarray(A); n=A.shape[0]
    if pos is None:
        ang=np.linspace(0,2*np.pi,n,endpoint=False); pos=np.c_[np.cos(ang),np.sin(ang)]
    plt.figure(figsize=(4,4))
    for i in range(n):
        for j in range(i+1,n):
            if A[i,j]!=0: plt.plot([pos[i,0],pos[j,0]],[pos[i,1],pos[j,1]],c='0.75',lw=1+2*abs(A[i,j]))
    c=values if values is not None else np.arange(n)
    plt.scatter(pos[:,0],pos[:,1],c=c,s=320,cmap='viridis',edgecolor='k',zorder=3)
    for i,(x,y) in enumerate(pos): plt.text(x,y,str(i),ha='center',va='center',color='white',weight='bold')
    plt.axis('off'); plt.title(title)
print('setup ok')

## A graph autoencoder learns node embeddings by reconstructing edges
A GAE compresses nodes into latent vectors Z and decodes links with sigmoid dot products. The examples compute scores, probabilities, binary cross-entropy, thresholded reconstruction, and negative sampling.

In [ ]:
# 1) inner-product decoder scores candidate edges
Z=np.array([[1.,0.],[0.8,0.2],[-1.,0.],[0.,1.]]); S=Z@Z.T
plt.figure(figsize=(4,3)); plt.imshow(S,cmap='coolwarm'); plt.colorbar(); plt.title('decoder scores Z Z^T')
assert abs(S[0,1]-0.8)<1e-9

In [ ]:
# 2) sigmoid maps scores to edge probabilities
Z=np.array([[1.,0.],[0.8,0.2],[-1.,0.],[0.,1.]]); P=sigmoid(Z@Z.T)
plt.figure(figsize=(4,3)); plt.imshow(P,cmap='viridis'); plt.colorbar(); plt.title('edge probabilities')
assert abs(P[0,1]-0.68997448)<1e-6

In [ ]:
# 3) positive and negative edge reconstruction loss
p_pos=sigmoid(0.8); p_neg=sigmoid(-1.0); loss=-(math.log(p_pos)+math.log(1-p_neg))/2
plt.figure(figsize=(5,3)); plt.bar(['p(pos)','p(neg)','BCE'],[p_pos,p_neg,loss]); plt.title('GAE reconstruction terms')
assert abs(loss-0.342181)<1e-5

In [ ]:
# 4) threshold probabilities to reconstruct adjacency
Z=np.array([[1.,0.],[0.8,0.2],[-1.,0.],[0.,1.]]); P=sigmoid(Z@Z.T); R=(P>0.6).astype(int); np.fill_diagonal(R,0)
plt.figure(figsize=(4,3)); plt.imshow(R,cmap='Greys'); plt.title('thresholded reconstruction')
assert R[0,1]==1 and R[0,2]==0

In [ ]:
# 5) negative sampling keeps training small
all_pairs=[(i,j) for i in range(4) for j in range(i+1,4)]; pos={(0,1),(2,3)}; neg=[e for e in all_pairs if e not in pos][:2]
plt.figure(figsize=(5,3)); plt.bar(['positives','sampled negatives'],[len(pos),len(neg)]); plt.title('balanced edge mini-batch')
assert len(neg)==2 and neg[0]==(0,2)